In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 120)

In [2]:
from pathlib import Path
from nih.paths import ensure_dirs, CORPUS_DIR

CORPUS_PATH = CORPUS_DIR / 'corpus.parquet'
df = pd.read_parquet(CORPUS_PATH)

In [3]:
df.shape
df.dtypes

APPLICATION_ID         int64
FY                     Int64
IC_NAME               object
ACTIVITY              object
PROJECT_TITLE         object
ABSTRACT_TEXT         object
PROJECT_TERMS         object
NIH_SPENDING_CATS     object
TOTAL_COST           float64
text                  object
dtype: object

## 1. Corpus shape & identity

In [4]:
df.head(3)

,APPLICATION_ID,FY,IC_NAME,ACTIVITY,PROJECT_TITLE,ABSTRACT_TEXT,PROJECT_TERMS,NIH_SPENDING_CATS,TOTAL_COST,text
0,8676197,2020,Veterans Affairs,IK2,Improving Care for PTSD,DESCRIPTION (provided by applicant): BACKGROUND & RATIONALE: Posttraumatic stress disorder (PTSD) is an often ...,Affect;Area;Award;career;career development;Caring;Chronic;Clinical;clinical practice;Code;design;Development;disabi...,None,NaN,Improving Care for PTSD\n\nDESCRIPTION (provided by applicant): BACKGROUND & RATIONALE: Posttraumatic stress d...
1,8785548,2020,Veterans Affairs,I01,Design and Evaluation of User Centered Electronic Health Records,"DESCRIPTION (provided by applicant): To date, there is little quantitative research on how providers use Electr...",acronyms;Address;Affect;Archives;Area;Automation;Back;base;care providers;Cities;Clinical;Code;Complement;Computer s...,None,NaN,Design and Evaluation of User Centered Electronic Health Records\n\nDESCRIPTION (provided by applicant): To dat...
2,8867710,2020,Veterans Affairs,I01,Comparative Effectiveness of Delivery Methods for Caregiver Support and Education,? DESCRIPTION (provided by applicant): Background Existing interventions for informal caregivers (CGs) of c...,18 year old;Address;Adult;Adult Children;Age;Aging;Ambulatory Care Facilities;Area;base;Budgets;care providers;care ...,None,NaN,Comparative Effectiveness of Delivery Methods for Caregiver Support and Education\n\n? DESCRIPTION (provided by ...


### 1.1 Corpus Size & primary key integrity

In [5]:
n_rows = len(df)
n_unique_ids = df["APPLICATION_ID"].nunique(dropna=True)
n_missing_ids = df["APPLICATION_ID"].isna().sum()

pd.Series({
    "n_rows": n_rows,
    "n_unique_APPLICATION_ID": n_unique_ids,
    "n_missing_APPLICATION_ID": n_missing_ids,
    "duplication_rate": 1 - (n_unique_ids / n_rows),
})

n_rows                      403988.0
n_unique_APPLICATION_ID     403988.0
n_missing_APPLICATION_ID         0.0
duplication_rate                 0.0
dtype: float64

In [6]:
assert n_missing_ids == 0, f"Found {n_missing_ids} missing APPLICATION_ID values."
assert n_unique_ids == n_rows, (
    f"APPLICATION_ID is not unique per row: {n_unique_ids} unique IDs vs {n_rows} rows."
)
print("✅ PK integrity OK: APPLICATION_ID is complete and unique.")

✅ PK integrity OK: APPLICATION_ID is complete and unique.


### 1.2 Temporal Coverage (FY)

In [7]:
fy_counts = df["FY"].value_counts(dropna=False).sort_index()
fy_counts

FY
2020    77835
2021    81338
2022    83140
2023    84082
2024    77593
Name: count, dtype: Int64

In [8]:
n_missing_fy = df["FY"].isna().sum()
unique_fy = sorted(df["FY"].dropna().unique().tolist())

pd.Series({
    "n_missing_FY": n_missing_fy,
    "n_unique_FY": len(unique_fy),
    "FY_min": min(unique_fy) if unique_fy else None,
    "FY_max": max(unique_fy) if unique_fy else None,
})

n_missing_FY       0
n_unique_FY        5
FY_min          2020
FY_max          2024
dtype: int64

In [9]:
(fy_counts / n_rows).rename("share").to_frame()

,share
FY,
2020,0.192667
2021,0.201338
2022,0.205798
2023,0.20813
2024,0.192068


In [10]:
assert n_missing_fy == 0, f"Found {n_missing_fy} missing FY values."
assert len(unique_fy) == 5, f"Expected 5 FY values, found {len(unique_fy)}: {unique_fy}"
print(f"✅ FY coverage OK: {unique_fy}")

✅ FY coverage OK: [2020, 2021, 2022, 2023, 2024]


### 1.3 Institutional distribution (IC_NAME)

In [11]:
ic_counts = df["IC_NAME"].value_counts(dropna=False)
ic_counts.head(15)

IC_NAME
NATIONAL CANCER INSTITUTE                                                        61891
NATIONAL INSTITUTE OF ALLERGY AND INFECTIOUS DISEASES                            43348
NATIONAL INSTITUTE OF GENERAL MEDICAL SCIENCES                                   42753
NATIONAL HEART, LUNG, AND BLOOD INSTITUTE                                        33689
NATIONAL INSTITUTE ON AGING                                                      30181
NATIONAL INSTITUTE OF DIABETES AND DIGESTIVE AND KIDNEY DISEASES                 25637
NATIONAL INSTITUTE OF NEUROLOGICAL DISORDERS AND STROKE                          25480
NATIONAL INSTITUTE OF MENTAL HEALTH                                              18846
EUNICE KENNEDY SHRIVER NATIONAL INSTITUTE OF CHILD HEALTH & HUMAN DEVELOPMENT    17949
NATIONAL INSTITUTE ON DRUG ABUSE                                                 13398
Veterans Affairs                                                                 10466
NATIONAL EYE INSTITUTE             

In [12]:
n_missing_ic = df["IC_NAME"].isna().sum()
n_unique_ic = df["IC_NAME"].nunique(dropna=True)

pd.Series({
    "n_missing_IC_NAME": n_missing_ic,
    "n_unique_IC_NAME": n_unique_ic,
})

n_missing_IC_NAME     0
n_unique_IC_NAME     40
dtype: int64

In [13]:
top_ic = (ic_counts.head(15) / n_rows * 100).round(2).rename("% of rows")
top_ic.to_frame()

,% of rows
IC_NAME,
NATIONAL CANCER INSTITUTE,15.32
NATIONAL INSTITUTE OF ALLERGY AND INFECTIOUS DISEASES,10.73
NATIONAL INSTITUTE OF GENERAL MEDICAL SCIENCES,10.58
"NATIONAL HEART, LUNG, AND BLOOD INSTITUTE",8.34
NATIONAL INSTITUTE ON AGING,7.47
NATIONAL INSTITUTE OF DIABETES AND DIGESTIVE AND KIDNEY DISEASES,6.35
NATIONAL INSTITUTE OF NEUROLOGICAL DISORDERS AND STROKE,6.31
NATIONAL INSTITUTE OF MENTAL HEALTH,4.66
EUNICE KENNEDY SHRIVER NATIONAL INSTITUTE OF CHILD HEALTH & HUMAN DEVELOPMENT,4.44


In [14]:
assert n_missing_ic == 0, f"Found {n_missing_ic} missing IC_NAME values."
print("✅ IC_NAME coverage OK (no missing values).")

✅ IC_NAME coverage OK (no missing values).


### 1.4 Abstract presence & length

In [15]:
abs_col = "ABSTRACT_TEXT"

n_missing_abs = df[abs_col].isna().sum()
n_empty_abs = (df[abs_col].fillna("").str.strip() == "").sum()

pd.Series({
    "n_missing_ABSTRACT_TEXT": n_missing_abs,
    "n_empty_ABSTRACT_TEXT": n_empty_abs,
    "% empty": (n_empty_abs / n_rows) if n_rows else None,
})

n_missing_ABSTRACT_TEXT    0.0
n_empty_ABSTRACT_TEXT      0.0
% empty                    0.0
dtype: float64

In [16]:
abs_len = df[abs_col].fillna("").str.len()

abs_len.describe(percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

count    403988.000000
mean       2681.335248
std         764.698515
min           1.000000
1%          503.000000
5%         1344.000000
10%        1728.000000
25%        2301.000000
50%        2828.000000
75%        3139.000000
90%        3257.000000
95%        3337.000000
99%        4423.130000
max       24993.000000
Name: ABSTRACT_TEXT, dtype: float64

In [17]:
# Shortest non-empty abstracts
short_idx = df[df[abs_col].fillna("").str.strip().ne("")][abs_col].fillna("").str.len().nsmallest(20).index
df.loc[short_idx, ["APPLICATION_ID", "FY", "IC_NAME", "PROJECT_TITLE", abs_col]].assign(
    ABSTRACT_LEN=lambda d: d[abs_col].fillna("").str.len()
)

,APPLICATION_ID,FY,IC_NAME,PROJECT_TITLE,ABSTRACT_TEXT,ABSTRACT_LEN
305940,10713143,2023,NATIONAL INSTITUTE OF GENERAL MEDICAL SCIENCES,Tackling Big Data problems in biomedical sciences with extended similarity methods,N,1
388306,10931404,2024,NATIONAL INSTITUTE OF GENERAL MEDICAL SCIENCES,Tackling Big Data problems in biomedical sciences with extended similarity methods,N,1
147097,10329723,2021,NATIONAL INSTITUTE ON DRUG ABUSE,MANAGEMENT OF THE PRIMATE AGING DATABASE,Aging,5
147754,10346910,2021,NATIONAL INSTITUTE OF MENTAL HEALTH,Cerebellum and cerebellar-regulated circuit contribution to Fragile X Syndrome -,BLANK,5
157652,10488041,2021,"OFFICE OF THE DIRECTOR, NATIONAL INSTITUTES OF HEALTH",IFG::OT::IFG: ROUNDTABLE ON GENOMICS AND PRECISION HEALTH,Aging,5
158597,10494013,2021,"OFFICE OF THE DIRECTOR, NATIONAL INSTITUTES OF HEALTH",IFG::OT::IFG: PARTIAL SUPPORT FOR THE CORE ACTIVITIES OF THE COMMITTEE ON POPULATION(CPOP),Aging,5
158956,10505498,2021,"OFFICE OF THE DIRECTOR, NATIONAL INSTITUTES OF HEALTH",NONHUMAN PRIMATE MODEL SYSTEMS: STATE OF THE SCIENCE AND FUTURE NEEDS,Aging,5
158957,10505503,2021,NATIONAL INSTITUTE ON DRUG ABUSE,DEVELOPMENT AND MAINTENANCE OF A MULTIGENOTYPIC OF A MULTIGENOTYPIC AGED MOUSE COLONY PERIOD OF PERFORMANCE MARCH 14...,Aging,5
158958,10505509,2021,NATIONAL INSTITUTE ON DRUG ABUSE,"DEVELOPMENT AND MAINTENANCE OF A MULTIGENOTYPIC AGED RAT COLONY PERIOD OF PERFORMANCE JANUARY 20, 2021 - JANUARY 19,...",Aging,5
158962,10505573,2021,NATIONAL INSTITUTE ON AGING,"SELECTION, PRODUCTION, CHARACTERIZATION AND DISTRIBUTION OF CULTURED CELLS FOR RESEARCH ON AGING; PERIOD OF PERFORMANCE",Aging,5


In [18]:
s = df["ABSTRACT_TEXT"].fillna("").str.strip()
aging_mask = s.str.fullmatch(r"Aging", case=False)
aging_mask.mean(), aging_mask.sum()

def top_counts(col, mask, n=20):
    return (df.loc[mask, col].value_counts().head(n))

print(top_counts("FY", aging_mask))
print(top_counts("IC_NAME", aging_mask))
print(top_counts("ACTIVITY", aging_mask))

FY
2023    23
2021    10
2022     5
Name: count, dtype: Int64
IC_NAME
NATIONAL INSTITUTE ON DRUG ABUSE                         17
NATIONAL INSTITUTE ON AGING                              12
OFFICE OF THE DIRECTOR, NATIONAL INSTITUTES OF HEALTH     8
NATIONAL INSTITUTE OF ENVIRONMENTAL HEALTH SCIENCES       1
Name: count, dtype: int64
ACTIVITY
N01    27
Y01     6
N02     5
Name: count, dtype: int64


In [19]:
df.loc[aging_mask, ["PROJECT_TERMS", "NIH_SPENDING_CATS"]].head(20)

,PROJECT_TERMS,NIH_SPENDING_CATS
147097,Aging;Databases;Primates,Aging
157652,Aging;Genomics;Precision Health,Aging;Cancer Genomics;Genetics;Human Genome;Precision Medicine
158597,Aging;Population,Aging
158956,Aging;Biological Models;Future;nonhuman primate;Science,Aging;Eye Disease and Disorders of Vision
158957,aged;Aging;Development;Maintenance;Mus;Performance,Aging
158958,aged;Aging;Development;Maintenance;Performance;Rattus,Aging
158962,Aging;Cultured Cells;Performance;Production;Research,Aging;Clinical Research
158963,aged;Aging;Contractor;Evaluation;Health Status;Rodent,Aging
158964,aged;Aging;Development;Maintenance;Performance;Rodent;Tissues,Aging
158965,Aging;Agreement;research and development,Aging


In [20]:
df.loc[aging_mask, ["PROJECT_TITLE"]].head(20)

,PROJECT_TITLE
147097,MANAGEMENT OF THE PRIMATE AGING DATABASE
157652,IFG::OT::IFG: ROUNDTABLE ON GENOMICS AND PRECISION HEALTH
158597,IFG::OT::IFG: PARTIAL SUPPORT FOR THE CORE ACTIVITIES OF THE COMMITTEE ON POPULATION(CPOP)
158956,NONHUMAN PRIMATE MODEL SYSTEMS: STATE OF THE SCIENCE AND FUTURE NEEDS
158957,DEVELOPMENT AND MAINTENANCE OF A MULTIGENOTYPIC OF A MULTIGENOTYPIC AGED MOUSE COLONY PERIOD OF PERFORMANCE MARCH 14...
158958,"DEVELOPMENT AND MAINTENANCE OF A MULTIGENOTYPIC AGED RAT COLONY PERIOD OF PERFORMANCE JANUARY 20, 2021 - JANUARY 19,..."
158962,"SELECTION, PRODUCTION, CHARACTERIZATION AND DISTRIBUTION OF CULTURED CELLS FOR RESEARCH ON AGING; PERIOD OF PERFORMANCE"
158963,PROVIDE AN INDEPENDENT SURVEILLANCE OF THE HEALTH STATUS OF RODENTS MAINTAINED IN CONTRACTOR FACILITIES FOR THE AGED...
158964,DEVELOPMENT AND MAINTENANCE OF AN AGED RODENT TISSUE BANKPERIOD OF PERFORMANCE: 3/15/2021 - 3/14/2022
158965,DBSR R&D Agreements


In [21]:
df.assign(L=s.str.len()).sort_values("L").head(100)[["APPLICATION_ID","FY","IC_NAME","PROJECT_TITLE","ABSTRACT_TEXT","L"]]

,APPLICATION_ID,FY,IC_NAME,PROJECT_TITLE,ABSTRACT_TEXT,L
305940,10713143,2023,NATIONAL INSTITUTE OF GENERAL MEDICAL SCIENCES,Tackling Big Data problems in biomedical sciences with extended similarity methods,N,1
388306,10931404,2024,NATIONAL INSTITUTE OF GENERAL MEDICAL SCIENCES,Tackling Big Data problems in biomedical sciences with extended similarity methods,N,1
230289,10554226,2022,NATIONAL INSTITUTE ON AGING,"SELECTION, PRODUCTION, CHARACTERIZATION AND DISTRIBUTION OF CULTURED CELLS FOR RESEARCH ON AGING; PERIOD OF PERFORMANCE",Aging,5
326005,10945587,2023,NATIONAL INSTITUTE ON DRUG ABUSE,DEVELOPMENT AND MAINTENANCE OF A MULTIGENOTYPIC AGED MOUSE COLONY,Aging,5
314806,10788191,2023,NATIONAL INSTITUTE ON AGING,"SELECTION, PRODUCTION, CHARACTERIZATION AND DISTRIBUTION OF CULTURED CELLS FOR RESEARCH ON AGING; PERIOD OF PERFORMANCE",Aging,5
158962,10505573,2021,NATIONAL INSTITUTE ON AGING,"SELECTION, PRODUCTION, CHARACTERIZATION AND DISTRIBUTION OF CULTURED CELLS FOR RESEARCH ON AGING; PERIOD OF PERFORMANCE",Aging,5
158965,10505582,2021,NATIONAL INSTITUTE ON AGING,DBSR R&D Agreements,Aging,5
231959,10585902,2022,NATIONAL INSTITUTE ON DRUG ABUSE,DEVELOPMENT AND MAINTENANCE OF AN AGED RODENT TISSUE BANKPERIOD OF PERFORMANCE: 3/15/2021 - 3/14/2022,Aging,5
315884,10798095,2023,NATIONAL INSTITUTE ON DRUG ABUSE,DEVELOPMENT AND MAINTENANCE OF AN AGED RODENT TISSUE BANKPERIOD OF PERFORMANCE: 3/15/2021 - 3/14/2022,Aging,5
326009,10945854,2023,"OFFICE OF THE DIRECTOR, NATIONAL INSTITUTES OF HEALTH",FORUM ON TRAUMATIC BRAIN INJURY - Aging,Aging,5


In [22]:
long_idx = abs_len.nlargest(10).index
df.loc[long_idx, ["APPLICATION_ID", "FY", "IC_NAME", "PROJECT_TITLE", abs_col]].assign(
    ABSTRACT_LEN=lambda d: d[abs_col].fillna("").str.len()
)

,APPLICATION_ID,FY,IC_NAME,PROJECT_TITLE,ABSTRACT_TEXT,ABSTRACT_LEN
75546,10225972,2020,NATIONAL INSTITUTE ON DRUG ABUSE,NINDS RHOFED BPN REGULATORY AFFAIRS CONSULTING IGF::CL::IGF,NINDS Biotechnology Products and Biologics Regulatory Affairs Consulting Services STATEMENT OF WORK FINAL REVISED D...,24993
391413,10948266,2024,NATIONAL INSTITUTE OF NEUROLOGICAL DISORDERS AND STROKE,A Mobile Health Application to Detect Absence Seizures using Hyperventilation and Eye-Movement Recordings,Research Strategy\n1. Executive Summary of Predicate SBIR Phase I Grant and Team\nThe overall goal of our predicate ...,23001
72801,10169166,2020,NATIONAL INSTITUTE ON DRUG ABUSE,NINDS BPN/ Lowery Creek /CMC Consulting Services IGF::CL::IGF,NIH DRUG DISCOVERY AND DEVELOPMENT CONSULTING SERVICES STATEMENT OF WORK A. BACKGROUND The National Institutes o...,18109
69518,10126765,2020,NATIONAL INSTITUTE ON DRUG ABUSE,NINDS BPN DMPK Consulting - White Global Pharma Consultants LLC IGF::CL::IGF,NIH DRUG DISCOVERY AND DEVELOPMENT CONSULTING SERVICES STATEMENT OF WORK A. BACKGROUND The National Institutes of...,17898
155219,10475332,2021,NATIONAL INSTITUTE OF DIABETES AND DIGESTIVE AND KIDNEY DISEASES,Improving perioperative management to reduce postoperative acute kidney injury and long-term renal risk,Postoperative acute kidney injury (AKI) is a major cause of morbidity and mortality in patients who undergo intraabd...,16148
401123,11098835,2024,NATIONAL INSTITUTE ON MINORITY HEALTH AND HEALTH DISPARITIES,Factors Influencing Pediatric Asthma into Adulthood (FIPA2),A. Summary of Funded Parent Grant:\nFactors Influencing Pediatric Asthma into Adulthood (R01MD019027)\nThe prevalenc...,15638
240996,10704984,2022,"OFFICE OF THE DIRECTOR, NATIONAL INSTITUTES OF HEALTH",MEDICAL & PUBLIC HEALTH PREPAREDNESS FOR DISASTER & EMERGENCIES AND ACTION COLLABORATIVE ON DISASTER RESEARCH RESPONSE,Forum for Medical & Public Health Preparedness for Disaster & Emergencies and Action Collaborative.\n\nPolicy Contex...,15445
77429,10269862,2020,"OFFICE OF THE DIRECTOR, NATIONAL INSTITUTES OF HEALTH",MEDICAL & PUBLIC HEALTH PREPAREDNESS FOR DISASTER & EMERGENCIES AND ACTION COLLABORATIVE ON DISASTER RESEARCH RESPONSE,Forum for Medical & Public Health Preparedness for Disaster & Emergencies and Action Collaborative. Policy Context ...,15443
158547,10493043,2021,"OFFICE OF THE DIRECTOR, NATIONAL INSTITUTES OF HEALTH",MEDICAL & PUBLIC HEALTH PREPAREDNESS FOR DISASTER & EMERGENCIES AND ACTION COLLABORATIVE ON DISASTER RESEARCH RESPONSE,Forum for Medical & Public Health Preparedness for Disaster & Emergencies and Action Collaborative. Policy Context ...,15443
396687,11005623,2024,NATIONAL CANCER INSTITUTE,NRG Oncology Network Group Operations Center,"NCI CCCT Biomarker, Imaging, & QOL Studies Funding Program (BIQSFP)\nAnnual Progress Report\nStudy Number: NRG-RTOG ...",15298


In [23]:
assert n_missing_abs == 0, f"Found {n_missing_abs} missing ABSTRACT_TEXT values."
assert n_empty_abs == 0, f"Found {n_empty_abs} empty/whitespace ABSTRACT_TEXT values."
print("✅ Abstract presence OK: no missing/empty abstracts.")

✅ Abstract presence OK: no missing/empty abstracts.


### 1.5 Identity consistency checks

In [24]:
fy_per_id = df.groupby("APPLICATION_ID")["FY"].nunique()
fy_per_id.value_counts().head(10)

FY
1    403988
Name: count, dtype: int64

In [25]:
ic_per_id = df.groupby("APPLICATION_ID")["IC_NAME"].nunique()
ic_per_id.value_counts().head(10)

IC_NAME
1    403988
Name: count, dtype: int64

In [26]:
max_fy = int(fy_per_id.max())
max_ic = int(ic_per_id.max())

assert max_fy == 1, f"Some APPLICATION_ID map to multiple FY values (max={max_fy})."
assert max_ic == 1, f"Some APPLICATION_ID map to multiple IC_NAME values (max={max_ic})."

print("✅ Identity consistency OK: each APPLICATION_ID maps to exactly one FY and one IC_NAME.")

✅ Identity consistency OK: each APPLICATION_ID maps to exactly one FY and one IC_NAME.


### 1.6 Metadata availability & basic numeric sanity

In [27]:
df["ACTIVITY"].value_counts(dropna=False).head(25)

ACTIVITY
R01    148438
R21     20706
P30     19628
R35     15012
P01     11983
U01     11513
F31     10463
U54      9803
T32      8975
ZIA      8101
P50      7476
U19      7282
I01      7138
K23      6681
K01      5889
P20      5708
K08      5354
R44      5137
F32      4998
R03      4486
N01      4448
R25      4264
F30      4149
R00      3932
K99      3523
Name: count, dtype: int64

In [28]:
def coverage(col: str) -> pd.Series:
    s = df[col]
    missing = s.isna().sum()
    empty = (s.fillna("").astype(str).str.strip() == "").sum()
    return pd.Series({
        "missing": missing,
        "empty_or_blank": empty,
        "present": n_rows - missing - empty,
        "present_rate": (n_rows - missing - empty) / n_rows if n_rows else None
    })

for col in ["PROJECT_TITLE", "PROJECT_TERMS", "NIH_SPENDING_CATS"]:
    display(pd.DataFrame({col: coverage(col)}))

,PROJECT_TITLE
missing,1.000000
empty_or_blank,1.000000
present,403986.000000
present_rate,0.999995


,PROJECT_TERMS
missing,8007.00000
empty_or_blank,8007.00000
present,387974.00000
present_rate,0.96036


,NIH_SPENDING_CATS
missing,41617.000000
empty_or_blank,41617.000000
present,320754.000000
present_rate,0.793969


In [32]:
cost = df["TOTAL_COST"]

pd.Series({
    "dtype": str(cost.dtype),
    "n_missing": cost.isna().sum(),
    "min": cost.min(skipna=True),
    "median": cost.median(skipna=True),
    "mean": cost.mean(skipna=True),
    "max": cost.max(skipna=True),
    "n_zero": (cost.fillna(0) == 0).sum(),
    "% zero_or_missing": ((cost.isna().sum() + (cost.fillna(0) == 0).sum()) / n_rows),
    "present_rate": (n_rows - cost.isna().sum() - (cost.fillna(0) == 0).sum()) / n_rows if n_rows else None
})

dtype                     float64
n_missing                   66523
min                           0.0
median                   390000.0
mean                 583426.88365
max                   417201451.0
n_zero                      66897
% zero_or_missing        0.330257
present_rate             0.669743
dtype: object

## Section 2. Missingness & Structural Weakness

### 2.1 Define placeholder & structural flags (read-only)

In [33]:
PLACEHOLDER_ABSTRACTS = {
    "aging",
    "alcohol",
    "no text",
}

df["_abstract_norm"] = df["ABSTRACT_TEXT"].str.strip().str.lower()

df["is_placeholder_abstract"] = df["_abstract_norm"].isin(PLACEHOLDER_ABSTRACTS)

df["is_placeholder_abstract"].value_counts()


is_placeholder_abstract
False    403890
True         98
Name: count, dtype: int64

In [34]:
df["abstract_len"] = df["ABSTRACT_TEXT"].str.len()

df["is_short_abstract"] = df["abstract_len"] < 500  # conservative threshold

pd.Series({
    "placeholder_rate": df["is_placeholder_abstract"].mean(),
    "short_abstract_rate": df["is_short_abstract"].mean(),
})


placeholder_rate       0.000243
short_abstract_rate    0.009884
dtype: float64

### 2.2 Missingness overview

In [35]:
def missingness(col):
    return pd.Series({
        "missing": df[col].isna().sum(),
        "missing_rate": df[col].isna().mean()
    })

cols_to_check = [
    "ABSTRACT_TEXT",
    "PROJECT_TITLE",
    "PROJECT_TERMS",
    "NIH_SPENDING_CATS",
    "TOTAL_COST",
]

pd.DataFrame({col: missingness(col) for col in cols_to_check}).T

,missing,missing_rate
ABSTRACT_TEXT,0.0,0.000000
PROJECT_TITLE,1.0,0.000002
PROJECT_TERMS,8007.0,0.019820
NIH_SPENDING_CATS,41617.0,0.103015
TOTAL_COST,66523.0,0.164666


**Effective missingness definition**

For text modeling purposes, we treat the following as *effectively missing abstracts*:
- null or empty strings
- ontology-level placeholder values (e.g. "Aging")

In [36]:
df["abstract_effectively_missing"] = (
    df["ABSTRACT_TEXT"].isna()
    | df["is_placeholder_abstract"]
)

pd.Series({
    "true_missing": df["ABSTRACT_TEXT"].isna().mean(),
    "placeholder": df["is_placeholder_abstract"].mean(),
    "effective_missing": df["abstract_effectively_missing"].mean(),
})

true_missing         0.000000
placeholder          0.000243
effective_missing    0.000243
dtype: float64

### 2.3 Temporal structure of missingness

In [37]:
(
    df.groupby("FY")["abstract_effectively_missing"]
    .mean()
    .rename("effective_missing_rate")
    .to_frame()
)

,effective_missing_rate
FY,
2020,0.000077
2021,0.000123
2022,0.000565
2023,0.000369
2024,0.000052


In [38]:
(
    df.groupby("FY")["is_placeholder_abstract"]
    .mean()
    .rename("placeholder_rate")
    .to_frame()
)

,placeholder_rate
FY,
2020,0.000077
2021,0.000123
2022,0.000565
2023,0.000369
2024,0.000052


### 2.4 Institutional structure (IC_NAME)

In [39]:
ic_missing = (
    df.groupby("IC_NAME")["abstract_effectively_missing"]
    .mean()
    .sort_values(ascending=False)
)

ic_missing.head(15)


IC_NAME
OFFICE OF THE DIRECTOR, NATIONAL INSTITUTES OF HEALTH                            0.009219
NATIONAL INSTITUTE ON DRUG ABUSE                                                 0.001269
EUNICE KENNEDY SHRIVER NATIONAL INSTITUTE OF CHILD HEALTH & HUMAN DEVELOPMENT    0.001003
NATIONAL INSTITUTE ON AGING                                                      0.000398
NATIONAL HEART, LUNG, AND BLOOD INSTITUTE                                        0.000297
NATIONAL INSTITUTE ON ALCOHOL ABUSE AND ALCOHOLISM                               0.000145
NATIONAL INSTITUTE OF ENVIRONMENTAL HEALTH SCIENCES                              0.000119
NATIONAL CANCER INSTITUTE                                                        0.000016
FOOD AND DRUG ADMINISTRATION                                                     0.000000
FOGARTY INTERNATIONAL CENTER                                                     0.000000
Center for Global Health                                                         0.000000
CL

In [40]:
(
    df.groupby("IC_NAME")["is_placeholder_abstract"]
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

IC_NAME
OFFICE OF THE DIRECTOR, NATIONAL INSTITUTES OF HEALTH                            0.009219
NATIONAL INSTITUTE ON DRUG ABUSE                                                 0.001269
EUNICE KENNEDY SHRIVER NATIONAL INSTITUTE OF CHILD HEALTH & HUMAN DEVELOPMENT    0.001003
NATIONAL INSTITUTE ON AGING                                                      0.000398
NATIONAL HEART, LUNG, AND BLOOD INSTITUTE                                        0.000297
NATIONAL INSTITUTE ON ALCOHOL ABUSE AND ALCOHOLISM                               0.000145
NATIONAL INSTITUTE OF ENVIRONMENTAL HEALTH SCIENCES                              0.000119
NATIONAL CANCER INSTITUTE                                                        0.000016
FOOD AND DRUG ADMINISTRATION                                                     0.000000
FOGARTY INTERNATIONAL CENTER                                                     0.000000
Center for Global Health                                                         0.000000
CL

### 2.5 Activity codes: documentation & structure

In [41]:
activity_counts = df["ACTIVITY"].value_counts()
activity_counts.head(20)

ACTIVITY
R01    148438
R21     20706
P30     19628
R35     15012
P01     11983
U01     11513
F31     10463
U54      9803
T32      8975
ZIA      8101
P50      7476
U19      7282
I01      7138
K23      6681
K01      5889
P20      5708
K08      5354
R44      5137
F32      4998
R03      4486
Name: count, dtype: int64

In [42]:
RARE_THRESHOLD = 50

rare_activity_codes = activity_counts[activity_counts < RARE_THRESHOLD].index

pd.Series({
    "n_unique_ACTIVITY": activity_counts.size,
    "n_rare_ACTIVITY": len(rare_activity_codes),
    "% rows in rare ACTIVITY": activity_counts[activity_counts < RARE_THRESHOLD].sum() / len(df)
})

n_unique_ACTIVITY          160.000000
n_rare_ACTIVITY             40.000000
% rows in rare ACTIVITY      0.002364
dtype: float64

In [43]:
placeholder_by_activity = (
    df.groupby("ACTIVITY")["is_placeholder_abstract"]
    .mean()
    .sort_values(ascending=False)
)

placeholder_by_activity[placeholder_by_activity > 0]

ACTIVITY
OT3    0.069767
OT2    0.063776
Y01    0.013436
N01    0.006969
N02    0.005952
N43    0.003636
Name: is_placeholder_abstract, dtype: float64

Only a small number of ACTIVITY codes exhibit non-zero placeholder abstract rates.
These mechanisms are responsible for the vast majority of abstract unavailability.

ACTIVITY × FY interaction analysis was attempted but showed no additional structure beyond ACTIVITY alone.
Placeholder abstracts are sparse and mechanism-specific rather than time–mechanism interactive.

### 2.6 Are placeholders just metadata leakage?

In [44]:
df.loc[df["is_placeholder_abstract"], "NIH_SPENDING_CATS"].value_counts().head(10)

NIH_SPENDING_CATS
Aging                                                                                     29
Health Disparities;Minority Health;Precision Medicine;Women's Health                       6
Alcoholism, Alcohol Use and Health;Substance Abuse                                         4
Aging;Clinical Research                                                                    3
Precision Medicine;Sleep Research                                                          2
Precision Medicine;Women's Health                                                          2
Precision Medicine                                                                         2
Data Science;Networking and Information Technology R&D (NITRD)                             2
Aging;Cancer Genomics;Genetics;Human Genome;Precision Medicine                             1
Clinical Research;Health Disparities;Minority Health;Precision Medicine;Women's Health     1
Name: count, dtype: int64

In [45]:
df.loc[df["is_placeholder_abstract"], "PROJECT_TERMS"].notna().mean()

np.float64(1.0)

### 2.7 Structural risk summary table

In [46]:
pd.DataFrame({
    "effective_missing_abstract": df["abstract_effectively_missing"].mean(),
    "placeholder_abstract": df["is_placeholder_abstract"].mean(),
    "short_abstract": df["is_short_abstract"].mean(),
    "rare_ACTIVITY": df["ACTIVITY"].isin(rare_activity_codes).mean(),
}, index=["rate"]).T

,rate
effective_missing_abstract,0.000243
placeholder_abstract,0.000243
short_abstract,0.009884
rare_ACTIVITY,0.002364


## Section 3. Subsections (clean & bounded)

### 3.1 Define modeling-eligable abstracts

In [47]:
# Assumes these were defined in Section 2
assert "abstract_effectively_missing" in df.columns
assert "is_placeholder_abstract" in df.columns
assert "abstract_len" in df.columns
assert "ACTIVITY" in df.columns

In [48]:
# From Section 2
placeholder_by_activity = (
    df.groupby("ACTIVITY")["is_placeholder_abstract"]
    .mean()
)

PROBLEMATIC_ACTIVITY_CODES = placeholder_by_activity[
    placeholder_by_activity > 0
].index.tolist()

len(PROBLEMATIC_ACTIVITY_CODES), PROBLEMATIC_ACTIVITY_CODES

(6, ['N01', 'N02', 'N43', 'OT2', 'OT3', 'Y01'])

In [49]:
MIN_ABSTRACT_LEN = 500  # conservative, empirically justified

df["eligible_for_text_modeling"] = (
    (~df["abstract_effectively_missing"]) &
    (~df["ACTIVITY"].isin(PROBLEMATIC_ACTIVITY_CODES)) &
    (df["abstract_len"] >= MIN_ABSTRACT_LEN)
)

df["eligible_for_text_modeling"].value_counts()

eligible_for_text_modeling
True     395446
False      8542
Name: count, dtype: int64

In [50]:
pd.Series({
    "eligible_rate": df["eligible_for_text_modeling"].mean(),
    "eligible_n": df["eligible_for_text_modeling"].sum(),
    "ineligible_n": (~df["eligible_for_text_modeling"]).sum(),
})

eligible_rate         0.978856
eligible_n       395446.000000
ineligible_n       8542.000000
dtype: float64

### 3.2 Sensitivity variants

### Modeling variants

The following modeling variants are defined for sensitivity analysis.
These are *labels* only; filtering is applied downstream.

- **base**: Eligible abstracts only (default)
- **include_problematic_activity**: Ignore ACTIVITY-based exclusion
- **include_short_abstracts**: Lower length threshold
- **exclude_recent_FY**: Remove most recent fiscal year(s)

In [51]:
df["variant_base"] = df["eligible_for_text_modeling"]

df["variant_include_problematic_activity"] = (
    (~df["abstract_effectively_missing"]) &
    (df["abstract_len"] >= MIN_ABSTRACT_LEN)
)

df["variant_include_short_abstracts"] = (
    (~df["abstract_effectively_missing"]) &
    (~df["ACTIVITY"].isin(PROBLEMATIC_ACTIVITY_CODES))
)

RECENT_FY = df["FY"].max()

df["variant_exclude_recent_FY"] = (
    df["eligible_for_text_modeling"] &
    (df["FY"] < RECENT_FY)
)

In [52]:
variant_cols = [
    "variant_base",
    "variant_include_problematic_activity",
    "variant_include_short_abstracts",
    "variant_exclude_recent_FY",
]

df[variant_cols].mean().to_frame("share").join(
    df[variant_cols].sum().to_frame("n")
)

,share,n
variant_base,0.978856,395446
variant_include_problematic_activity,0.990116,399995
variant_include_short_abstracts,0.982477,396909
variant_exclude_recent_FY,0.789514,318954


### 3.3 Corpus subdeclarations

In [53]:
ic_raw = df["IC_NAME"].astype("string")

ic_norm = (
    ic_raw
    .fillna("")
    .str.replace("\u00A0", " ", regex=False)   # non-breaking space
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)     # collapse runs of whitespace
    .str.upper()
)

# sanity peek
ic_norm.value_counts().head(20)

IC_NAME
NATIONAL CANCER INSTITUTE                                                        61891
NATIONAL INSTITUTE OF ALLERGY AND INFECTIOUS DISEASES                            43348
NATIONAL INSTITUTE OF GENERAL MEDICAL SCIENCES                                   42753
NATIONAL HEART, LUNG, AND BLOOD INSTITUTE                                        33689
NATIONAL INSTITUTE ON AGING                                                      30181
NATIONAL INSTITUTE OF DIABETES AND DIGESTIVE AND KIDNEY DISEASES                 25637
NATIONAL INSTITUTE OF NEUROLOGICAL DISORDERS AND STROKE                          25480
NATIONAL INSTITUTE OF MENTAL HEALTH                                              18846
EUNICE KENNEDY SHRIVER NATIONAL INSTITUTE OF CHILD HEALTH & HUMAN DEVELOPMENT    17949
NATIONAL INSTITUTE ON DRUG ABUSE                                                 13398
VETERANS AFFAIRS                                                                 10466
NATIONAL EYE INSTITUTE             

In [54]:
NEURO_IC_NAMES = {
    "National Institute of Neurological Disorders and Stroke",
    "National Institute of Mental Health",
    "National Institute on Drug Abuse",
    "National Institute on Alcohol Abuse and Alcoholism",
    "National Institute on Aging",
    "National Institute on Deafness and Other Communication Disorders",
    "National Eye Institute",
}
neuro_norm = {s.strip().upper() for s in NEURO_IC_NAMES}

df["is_neuro_ic"] = ic_norm.isin(neuro_norm)
df["is_neuro_ic"].value_counts()

is_neuro_ic
False    293344
True     110644
Name: count, dtype: int64

In [55]:
# Lightweight keyword heuristic (diagnostic only)
NEURO_KEYWORDS = [
    "brain", "neuro", "cognitive", "synapse",
    "dementia", "alzheim", "parkinson",
]

df["is_neuro_keyword"] = (
    df["ABSTRACT_TEXT"]
    .str.lower()
    .str.contains("|".join(NEURO_KEYWORDS), regex=True)
)

df["is_neuro_keyword"].value_counts()

is_neuro_keyword
False    280381
True     123607
Name: count, dtype: int64

In [56]:
pd.crosstab(
    df["is_neuro_ic"],
    df["is_neuro_keyword"],
    normalize="index"
)

is_neuro_keyword,False,True
is_neuro_ic,,
False,0.843723,0.156277
True,0.297169,0.702831


### 3.4 Cardinality checks

In [57]:
(
    df[df["eligible_for_text_modeling"]]
    .groupby("FY")
    .size()
    .rename("eligible_n")
    .to_frame()
)

,eligible_n
FY,
2020,76055
2021,79533
2022,81217
2023,82149
2024,76492


In [58]:
(
    df[df["eligible_for_text_modeling"]]
    .groupby("IC_NAME")
    .size()
    .sort_values(ascending=False)
    .head(15)
)

IC_NAME
NATIONAL CANCER INSTITUTE                                                        60164
NATIONAL INSTITUTE OF GENERAL MEDICAL SCIENCES                                   42448
NATIONAL INSTITUTE OF ALLERGY AND INFECTIOUS DISEASES                            40914
NATIONAL HEART, LUNG, AND BLOOD INSTITUTE                                        32997
NATIONAL INSTITUTE ON AGING                                                      30081
NATIONAL INSTITUTE OF DIABETES AND DIGESTIVE AND KIDNEY DISEASES                 25505
NATIONAL INSTITUTE OF NEUROLOGICAL DISORDERS AND STROKE                          25402
NATIONAL INSTITUTE OF MENTAL HEALTH                                              18777
EUNICE KENNEDY SHRIVER NATIONAL INSTITUTE OF CHILD HEALTH & HUMAN DEVELOPMENT    17229
NATIONAL INSTITUTE ON DRUG ABUSE                                                 12623
Veterans Affairs                                                                 10432
NATIONAL EYE INSTITUTE             

### 3.5 Modeling handoff contract

### Modeling contract for `03_embeddings.ipynb`

- **Input rows** must satisfy: `variant_base == True`
- **Text field**: `ABSTRACT_TEXT`
- **No placeholder or effectively missing abstracts**
- **No structurally weak ACTIVITY mechanisms**
- **Minimum abstract length**: 500 characters
- Sensitivity analyses may use alternate `variant_*` flags
- No filtering logic should be reimplemented downstream